# LOF hyperparam selection

In [1]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [8]:
import time
import joblib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

from sdss.metadata import MetaData

meta = MetaData()

# Custom Functions

## Winner iForest

In [3]:
def pick_iforest_params(df, consensus_threshold):

    """
    Given a dataframe of boolean anomaly flags from different
    Isolation Forest models (with different hyperparameters),
    calculate which parameter set is the most stable, i.e. which
    parameter set's anomalies overlap the most with the
    consensus anomalies (those that at least `consensus_threshold`
    models agreed were anomalies).
    """

    assert consensus_threshold > 0
    assert consensus_threshold <= df.shape[1] - 1

    stability_scores = {}

    # Identify points that at least N of the models agreed were anomalies
    consensus_anomalies = df['consensus_score'] >= consensus_threshold

    for col in df.columns[:-1]: # Exclude the consensus_score column
        # How many of this model's anomalies are also 'consensus' anomalies?
        overlap = (df[col] & consensus_anomalies).sum()
        stability_scores[col] = overlap

    # The "Optimal" params are the ones with the highest overlap
    best_params = max(stability_scores, key=stability_scores.get)

    print(f"The most stable parameter set is: {best_params}")

    return stability_scores, best_params

## Scale data

In [4]:
def standard_scaler(latent_arr):

    scaler = StandardScaler()
    
    latent_scaled = scaler.fit_transform(latent_arr)
    
    return latent_scaled

# Config

## Directories

In [9]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
ch4_dir = f"{thesis_dir}/chapters/04_figures"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
models_dir = f"{data_dir}/models"
latent_dir = f"{data_dir}/latent"
bins_ids = [f'bin_{i:02d}' for i in range(4)] 

## Data

In [6]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1
n_wave = wave.shape

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)


## Latent per bin

In [45]:
latent_bin_dict = {}

for bin_id in bins_ids:

    latent_bin_dict[bin_id] = np.load(
        f"{latent_dir}/{bin_id}/latent_{bin_id}.npy"
    )

# Consensus based stability

In [ ]:
iforest_hyper_params_df_dict = {}

for bin_id in bins_ids:

    print(
        "Loading Isolation Forest hyperparameter"
        f"search results for {bin_id}")
        
    iforest_hyper_params_df_dict[bin_id] = pd.read_csv(
        f"{latent_dir}/{bin_id}/iforest/iforeslatent_bin_dict[bin_id]t_hypersearch_{bin_id}.csv",
    )

Loading Isolation Forest hyperparametersearch results for bin_00
Loading Isolation Forest hyperparametersearch results for bin_01
Loading Isolation Forest hyperparametersearch results for bin_02
Loading Isolation Forest hyperparametersearch results for bin_03


In [43]:
iforest_hyper_params_df_dict[bin_id].head()

,s64_e100_f100,s64_e100_f75,s64_e100_f50,s64_e200_f100,s64_e200_f75,s64_e200_f50,s64_e300_f100,s64_e300_f75,s64_e300_f50,s128_e100_f100,...,s512_e100_f100,s512_e100_f75,s512_e100_f50,s512_e200_f100,s512_e200_f75,s512_e200_f50,s512_e300_f100,s512_e300_f75,s512_e300_f50,consensus_score
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [44]:
stability_scores_dict = {}
best_params_dict = {}
# there are 36 models
consensus_threshold = 18

for bin_id, df in iforest_hyper_params_df_dict.items():

    print(f"Calculating stability scores for {bin_id}")

    stability_scores, best_params = pick_iforest_params(
        df=df.copy(),
        consensus_threshold=consensus_threshold
    )

    stability_scores_dict[bin_id] = stability_scores
    best_params_dict[bin_id] = best_params

Calculating stability scores for bin_00
The most stable parameter set is: s128_e300_f50
Calculating stability scores for bin_01
The most stable parameter set is: s256_e300_f75
Calculating stability scores for bin_02
The most stable parameter set is: s256_e300_f100
Calculating stability scores for bin_03
The most stable parameter set is: s128_e300_f100


In [31]:
best_params_dict

{'bin_00': 's128_e300_f50',
 'bin_01': 's256_e300_f75',
 'bin_02': 's256_e300_f100',
 'bin_03': 's128_e300_f100'}

In [30]:
bin_id = 'bin_02'
consensus_threshold = 18
df = iforest_hyper_params_df_dict[bin_id].copy()
consensus_mask = df['consensus_score'] > consensus_threshold
df.loc[consensus_mask, ['s256_e300_f100', 'consensus_score']].sum()


s256_e300_f100      1725
consensus_score    59800
dtype: int64

In [29]:
bin_id = 'bin_03'
stability_scores_dict[bin_id]
_df = pd.DataFrame.from_dict(stability_scores_dict[bin_id], orient='index', columns=['overlap_with_consensus'])
_df.sort_values('overlap_with_consensus', ascending=False)

,overlap_with_consensus
s128_e300_f100,1571
s512_e300_f50,1568
s256_e300_f50,1565
s256_e300_f100,1564
s256_e300_f75,1547
s128_e300_f50,1543
s512_e300_f75,1543
s256_e200_f75,1541
s256_e200_f100,1541
s512_e200_f50,1533


# Train and save models

In [39]:
models_dict = {}

# for bin_id in bins_ids:
for bin_id in bins_ids:

    latent_scaled = standard_scaler(
        latent_bin_dict[bin_id]
    )

    e, s, f = best_params_dict[bin_id].split('_')
    e, s, f = int(e[1:]), int(s[1:]), int(f[1:])/100

    model_fname = f"iforest_{bin_id}_{best_params_dict[bin_id]}.joblib"
    save_dir = os.path.join(
        latent_dir, bin_id, 'iforest'
    )
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, model_fname)

    start_time = time.perf_counter()

    iforest = IsolationForest(
        n_estimators=e,
        max_samples=s,
        max_features=f,
        contamination=0.01,
        n_jobs=-1,
        random_state=42
    )

    iforest.fit(latent_scaled)
    
    end_time = time.perf_counter()
    
    print(
        f"Trained Isolation Forest model for {bin_id} "
        f"in {end_time - start_time:.2f} seconds."
    )

    joblib.dump(iforest, save_path)
    models_dict[bin_id] = iforest

Trained Isolation Forest model for bin_00 in 1.26 seconds.
Trained Isolation Forest model for bin_01 in 2.76 seconds.
Trained Isolation Forest model for bin_02 in 2.00 seconds.
Trained Isolation Forest model for bin_03 in 1.03 seconds.


In [49]:
bin_id = 'bin_00'

model_fname = f"iforest_{bin_id}_{best_params_dict[bin_id]}.joblib"
print(f"Loading model from {model_fname}")
save_path = os.path.join(
    latent_dir, bin_id, 'iforest', model_fname
)
# Load the pre-trained model
loaded_iforest = joblib.load(save_path)

latent_arr = standard_scaler(
    latent_bin_dict[bin_id]
)
scores = loaded_iforest.decision_function(latent_arr)
scores.shape, scores.min(), scores.max()

Loading model from iforest_bin_00_s128_e300_f50.joblib


((181850,), -0.21287442039961868, 0.23269234646139153)

In [51]:
np.percentile(scores, 99)

0.21942804760523427